In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool

In [2]:
train_data = pd.read_csv('../data/model/full.csv')
train_labels = train_data['redemption_status'].values
train_data = train_data.drop(['id','customer_id','redemption_status'], axis=1)
valid_data = pd.read_csv('../data/model/valid.csv')
valid_labels = valid_data['redemption_status'].values
valid_data = valid_data.drop(['id','customer_id','redemption_status'], axis=1)
test_data = pd.read_csv('../data/model/test.csv')
score = test_data[['id']].copy()
test_data = test_data.drop(['id','customer_id'], axis=1)
print(train_data.shape, valid_data.shape)

(78369, 66) (22606, 66)


In [3]:
train_data = Pool(train_data, train_labels)
valid_data = Pool(valid_data, valid_labels)
test_data = Pool(test_data)

In [4]:
params = {}
params['num_boost_round'] = 1200
params['max_depth'] = 5
params['learning_rate'] = 0.01
params['colsample_bylevel'] = 0.5
params['reg_lambda'] = 0.01
params['eval_metric'] = 'AUC'

In [5]:
model = CatBoostClassifier(**params)

In [6]:
model.fit(train_data, eval_set=[train_data, valid_data], early_stopping_rounds=1500, verbose_eval=100)

0:	test: 0.8540032	test1: 0.8730463	best: 0.8730463 (0)	total: 76.7ms	remaining: 1m 31s
100:	test: 0.9430003	test1: 0.9422485	best: 0.9422485 (100)	total: 2.43s	remaining: 26.4s
200:	test: 0.9579886	test1: 0.9583602	best: 0.9583602 (200)	total: 4.72s	remaining: 23.5s
300:	test: 0.9649404	test1: 0.9661115	best: 0.9661115 (300)	total: 6.98s	remaining: 20.9s
400:	test: 0.9678585	test1: 0.9690341	best: 0.9690341 (400)	total: 9.26s	remaining: 18.5s
500:	test: 0.9695656	test1: 0.9708400	best: 0.9708400 (500)	total: 11.6s	remaining: 16.1s
600:	test: 0.9711851	test1: 0.9725906	best: 0.9725906 (600)	total: 13.8s	remaining: 13.8s
700:	test: 0.9725132	test1: 0.9739244	best: 0.9739244 (700)	total: 16.1s	remaining: 11.5s
800:	test: 0.9739702	test1: 0.9753474	best: 0.9753474 (800)	total: 18.4s	remaining: 9.17s
900:	test: 0.9754646	test1: 0.9769957	best: 0.9769957 (900)	total: 20.7s	remaining: 6.86s
1000:	test: 0.9768168	test1: 0.9783900	best: 0.9783900 (1000)	total: 22.9s	remaining: 4.56s
1100:	test

In [7]:
score['redemption_status'] = model.predict_proba(test_data)[:,-1]

In [8]:
score.to_csv('../data/score/score_v3.csv', index=False)